# Colab eval runner

Runs the jobs too heavy for the laptop, on Colab's free T4 GPU, with `qwen2.5:7b` under Ollama.

**Before you start:** Runtime > Change runtime type > T4 GPU. Add your key under Secrets (the key icon
on the left) as `GEMINI_API_KEY`, and allow this notebook to use it.

**Run the cells in order, one at a time.** Each one is a step you can rerun alone. After a reconnect,
run Steps 0 to 3 again (they skip what is already done), then the step you want.

Colab deletes everything when the runtime ends, so run Step 11 before you leave.

**Save without outputs.** Don't save this notebook back to GitHub with its outputs in it.

In [ ]:
# Step 0: settings and helpers. Run first, and again after any restart. Changes nothing on disk.
import json, os, re, shutil, subprocess, sys, time, urllib.request, zipfile
from datetime import datetime, timezone
from pathlib import Path

BRANCH = "stage-4-streaming-followups"
REPO_URL = "https://github.com/aryanwaingankar4/shipment-intelligence-rag.git"
OLLAMA_MODEL = "qwen2.5:7b"
GEMINI_MODEL = "gemini-3.8-flash"  # saved answers are per model: changing this re-asks all of them
EVAL_RPM = "4"  # Gemini calls per minute; the free tier allows about 5
SAVE_TO_DRIVE = False  # True: Step 11 saves to Drive and Step 4 restores the newest zip from there
DRIVE_FOLDER = "shipment-rag-eval"
LAPTOP_DIGEST = "845dbda0ea48"  # the judge build behind the laptop's saved verdicts

ROOT = Path(os.environ.get("COLAB_RUNNER_ROOT", "/content"))
REPO = ROOT / "shipment-intelligence-rag"
VENV = ROOT / "venv"
LOGS = ROOT / "logs"
PY, CLI = VENV / "bin" / "python", VENV / "bin" / "shipment-rag"
OLLAMA_URL = "http://localhost:11434"
CACHE_PREFIX = "data/eval/.cache/"
CALIBRATION_MARK = LOGS / "calibration_passed"


def gemini_key():
    """Only from Colab Secrets, and only handed to the jobs that need it. Never print it."""
    from google.colab import userdata
    return userdata.get("GEMINI_API_KEY")


def job_env(with_gemini=False):
    env = {**os.environ, "JUDGE_PROVIDER": "ollama", "JUDGE_MODEL": OLLAMA_MODEL,
           "CITATION_CHECKER": "nli", "GEMINI_MODEL": GEMINI_MODEL, "REWRITE_PROVIDER": "off",
           "OLLAMA_BASE_URL": OLLAMA_URL, "LLM_MAX_RETRIES": "1", "PYTHONUTF8": "1",
           "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1",
           "TOKENIZERS_PARALLELISM": "false"}
    env.pop("GEMINI_API_KEY", None)
    if with_gemini:
        env["GEMINI_API_KEY"] = gemini_key()
    return env


def run(cmd, log_name, env=None, show=True):
    """Run a command in the repo, keeping its output in logs/. Returns the exit code."""
    LOGS.mkdir(parents=True, exist_ok=True)
    proc = subprocess.Popen([str(c) for c in cmd], cwd=REPO, env=env or job_env(), text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            encoding="utf-8", errors="replace")
    with open(LOGS / log_name, "w", encoding="utf-8") as log:
        for line in proc.stdout:
            log.write(line)
            log.flush()
            if show:
                print(line, end="")
    return proc.wait()


def ollama_api(path, body=None, timeout=300):
    data = None if body is None else json.dumps(body).encode()
    request = urllib.request.Request(OLLAMA_URL + path, data=data,
                                     headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=timeout) as reply:
        return json.load(reply)


def ensure_ollama():
    """Start the Ollama server if it isn't answering. Safe to call any number of times."""
    try:
        ollama_api("/api/tags", timeout=3)
        return
    except OSError:
        pass
    if not shutil.which("ollama"):
        raise RuntimeError("Ollama is not installed: run Step 3 first.")
    LOGS.mkdir(parents=True, exist_ok=True)
    subprocess.Popen(["ollama", "serve"], stdout=open(LOGS / "ollama.log", "a"),
                     stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(30):
        time.sleep(1)
        try:
            ollama_api("/api/tags", timeout=3)
            return
        except OSError:
            continue
    raise RuntimeError("Ollama did not start: see logs/ollama.log")


def drive_folder():
    from google.colab import drive
    drive.mount(str(ROOT / "drive"))
    folder = ROOT / "drive" / "MyDrive" / DRIVE_FOLDER
    folder.mkdir(parents=True, exist_ok=True)
    return folder


def restore_cache(zip_path):
    """Copy saved answers and verdicts from a zip into the repo. Takes nothing else from it."""
    restored = 0
    base = (REPO / CACHE_PREFIX).resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for name in archive.namelist():
            if not name.startswith(CACHE_PREFIX) or name.endswith("/"):
                continue
            target = (REPO / name).resolve()
            if base not in target.parents:
                raise RuntimeError(f"unsafe path in zip: {name}")
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(name))
            restored += 1
    return restored


print("Settings loaded. Gemini model:", GEMINI_MODEL)

In [ ]:
# Step 1: check for the GPU. If this fails: Runtime > Change runtime type > T4 GPU, then start again.
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True) if shutil.which("nvidia-smi") else None
if gpu is None or gpu.returncode != 0:
    raise RuntimeError("No GPU on this runtime. Runtime > Change runtime type > T4 GPU.")
print("GPU:", gpu.stdout.strip())

In [ ]:
# Step 2: get the code and install it in its own environment (about 5 minutes the first time).
# Its own environment avoids clashing with Colab's preinstalled packages. Skipped if already done.
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO)],
                   check=True)
if not CLI.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "venv", "--python", "3.12", str(VENV)], check=True)
    subprocess.run(["uv", "pip", "install", "--python", str(PY), "-e", str(REPO)], check=True)
commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
cuda = subprocess.run([str(PY), "-c", "import torch; print(torch.cuda.is_available())"],
                      capture_output=True, text=True).stdout.strip()
print(f"Code at {commit} on {BRANCH}. PyTorch sees the GPU: {cuda}")

In [ ]:
# Step 3: install Ollama, download the model and load it onto the GPU (about 5 minutes).
if not shutil.which("ollama"):
    subprocess.run(["apt-get", "update", "-qq"], check=True, stdout=subprocess.DEVNULL)
    subprocess.run(["apt-get", "install", "-y", "-qq", "zstd", "pciutils", "lshw"], check=True,
                   stdout=subprocess.DEVNULL)
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
ensure_ollama()
subprocess.run(["ollama", "pull", OLLAMA_MODEL], check=True)
ollama_api("/api/generate", {"model": OLLAMA_MODEL, "prompt": "ok", "stream": False,
                             "options": {"num_predict": 1}})
digest = next(m["digest"][:12] for m in ollama_api("/api/tags")["models"]
              if m["name"] == OLLAMA_MODEL)
loaded = ollama_api("/api/ps")["models"][0]
share = loaded.get("size_vram", 0) / max(loaded["size"], 1)
print(f"{OLLAMA_MODEL} digest {digest}, {share:.0%} on the GPU")
if share < 0.99:
    print("WARNING: not fully on the GPU, so this will be slow.")
if digest != LAPTOP_DIGEST:
    print(f"NOTE: the laptop's verdicts came from digest {LAPTOP_DIGEST}, so they won't be reused.")

In [ ]:
# Step 4: bring back saved answers, so Gemini isn't asked again.
# Pick eval_cache.zip from your Downloads, or a results zip from an earlier Colab run.
# With SAVE_TO_DRIVE on, the newest zip in Drive is used if there is one.
# No zip? Press Cancel: everything still works but uses more of the daily Gemini quota.
saved = None
if SAVE_TO_DRIVE:
    zips = sorted(drive_folder().glob("*.zip"))
    saved = zips[-1] if zips else None
if saved is None:
    from google.colab import files
    uploaded = files.upload()
    saved = Path(next(iter(uploaded))).resolve() if uploaded else None
print(f"Restored {restore_cache(saved)} saved files from {saved.name}" if saved
      else "No zip: starting with no saved answers.")

In [ ]:
# Step 5: build the search index, and check retrieval matches the laptop. No LLM, no quota.
code = run([CLI, "ingest", "--rebuild"], "ingest.log", show=False)
if code:
    raise RuntimeError("Indexing failed: see logs/ingest.log")
code = run([CLI, "eval"], "retrieval_eval.log")
print("\nRetrieval check:", "passed" if code == 0 else "BELOW THE GATE, tell Claude before going on")

In [ ]:
# Step 6: paraphrase eval (D5). Does rewriting questions help retrieval? Local model only, no Gemini.
ensure_ollama()
code = run([PY, "scripts/eval_rewrite.py", "--model", OLLAMA_MODEL,
            "--report", "data/eval/reports/stage4_rewrite.md"], "eval_rewrite.log")
print("\nD5 finished." if code == 0 else f"\nD5 FAILED (exit {code}): see logs/eval_rewrite.log")

In [ ]:
# Step 7: does the judge still catch every planted flaw on this GPU? Local model only, no Gemini.
# Saved verdicts are ignored here on purpose: a GPU can answer differently from the laptop's CPU.
ensure_ollama()
CALIBRATION_MARK.unlink(missing_ok=True)
code = run([CLI, "eval-gen", "--calibration-only", "--no-cache"], "calibration.log")
if code == 0:
    CALIBRATION_MARK.write_text(datetime.now(timezone.utc).isoformat())
    print("\nCalibration passed: Step 8 can run.")
else:
    print("\nCalibration FAILED: don't score the golden set. Save with Step 11 and tell Claude.")

In [ ]:
# Step 8: score the golden set. Uses Gemini for new answers and resumes from saved ones.
# Stops by itself on quota or repeated 503s, with everything so far kept. Rerun any time.
# Verdicts go to the report, not this output, so the hand-label check stays blind.
ensure_ollama()
if not CALIBRATION_MARK.exists():
    raise RuntimeError("Run Step 7 first: the judge must pass calibration on this GPU.")
code = run([CLI, "eval-gen", "--rpm", EVAL_RPM,
            "--report", "data/eval/reports/stage2_generation.md"],
           "eval_gen.log", env=job_env(with_gemini=True), show=False)
log = (LOGS / "eval_gen.log").read_text(encoding="utf-8", errors="replace")
report = REPO / "data/eval/reports/stage2_generation.md"
partial = ""
if report.exists():
    partial = next((l for l in report.read_text(encoding="utf-8").splitlines()
                    if "Partial run" in l), "")
print("Exit code:", code)
for line in log.splitlines():
    if line.startswith("Answer model"):
        print(line.split("; judge")[0])
print("503 errors:", len(re.findall(r"ServerError: 503", log)),
      "| quota errors:", log.count("llm quota exhausted"))
if code == 2:
    print("Gemini wasn't reachable, so nothing was scored. Try again later.")
elif code == 1:
    print("Nothing was scored: see logs/eval_gen.log.")
elif partial:
    print(partial.replace("*", "").strip("> "))
    print("Stopped early. Run Step 11 to save, then rerun this step later or tomorrow.")
else:
    print("All 26 questions scored.")

In [ ]:
# Step 9: prepare the blind hand-label file. Saved answers only: makes no Gemini calls.
# Holds cited sentences and their passages, with no checker verdicts.
code = run([PY, "scripts/prepare_citation_labels.py", "--out",
            "data/eval/citation_labels_todo.json"],
           "prepare_labels.log", env=job_env(with_gemini=True), show=False)
log = (LOGS / "prepare_labels.log").read_text(encoding="utf-8", errors="replace")
print(*[l for l in log.splitlines() if l.startswith(("answers to label", "written to"))], sep="\n")
if code:
    print(f"Exit {code}: no labelled answers yet, or see logs/prepare_labels.log.")

In [ ]:
# Step 10 (optional): list the Gemini flash models this key can use. Uses no generation quota.
# Only useful if gemini-3.8-flash keeps returning 503. To switch, change GEMINI_MODEL in Step 0.
code = run([CLI, "models"], "models.log", env=job_env(with_gemini=True), show=False)
names = (LOGS / "models.log").read_text(encoding="utf-8", errors="replace").splitlines()
print(*[n for n in names if "flash" in n], sep="\n")

In [ ]:
# Step 11: save everything to one zip, check it holds no key, then download it or copy it to Drive.
# Also unloads the model. Safe to rerun, and safe after a partial run.
stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
members = {}
for folder, prefix in [(REPO / "data/eval/.cache", CACHE_PREFIX),
                       (REPO / "data/eval/reports", "data/eval/reports/"),
                       (LOGS, "logs/")]:
    for path in sorted(folder.rglob("*")) if folder.exists() else []:
        if path.is_file() and path.name != "ollama.log":
            members[prefix + path.relative_to(folder).as_posix()] = path
labels = REPO / "data/eval/citation_labels_todo.json"
if labels.exists():
    members["data/eval/citation_labels_todo.json"] = labels

key = (gemini_key() or "").encode()
key_shaped = re.compile(rb"AIza[0-9A-Za-z_\-]{30,}")
for name, path in members.items():
    data = path.read_bytes()
    if (key and key in data) or key_shaped.search(data):
        raise RuntimeError(f"Not zipping: {name} contains an API key. Tell Claude. Nothing was saved.")

commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
info = {"finished_utc": stamp, "commit": commit, "branch": BRANCH, "gemini_model": GEMINI_MODEL,
        "judge_model": OLLAMA_MODEL, "machine": "Colab GPU"}
out = ROOT / f"colab_results_{stamp}.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as archive:
    for name, path in members.items():
        archive.write(path, name)
    archive.writestr("run_info.json", json.dumps(info, indent=1))
print(f"{out.name}: {len(members)} files, {out.stat().st_size / 1024:.0f} KB")

try:
    ensure_ollama()
    ollama_api("/api/generate", {"model": OLLAMA_MODEL, "keep_alive": 0})
    print("Model unloaded.")
except (OSError, RuntimeError):
    pass  # no Ollama here, or it's already gone: nothing to unload

if SAVE_TO_DRIVE:
    shutil.copy(out, drive_folder() / out.name)
    print(f"Saved to Google Drive: {DRIVE_FOLDER}/{out.name}")
else:
    try:
        from google.colab import files
    except ImportError:
        print("Not on Colab: zip left at", out)
    else:
        files.download(str(out))